# Gallery Matching / Recognition with MATA

This notebook demonstrates the **Gallery Matching** feature introduced in v1.9.5.
You will:

1. Build a `Gallery` by embedding representative fruit images with CLIP
2. Save and reload the gallery (`.npz`)
3. Query held-out images with `mata.run("recognize", ...)` and inspect `Matches` results
4. Run a batch evaluation across multiple fruit classes
5. See how to compose the pipeline as a graph

**Dataset used**: `data/fruit_classification/` — 33 labeled fruit varieties with a `train/` split (~300+ images each). The Kaggle `test/` split contains flat unlabelled images; held-out queries are taken from the tail of the `train/` split instead.  
**Model used**: `openai/clip-vit-base-patch32` (zero-shot visual encoder)  
**Prerequisites**: `pip install datamata[notebook]`


In [13]:
# Uncomment to install if needed:
# !pip install datamata[notebook]

## Dataset — Fruit Recognition (Kaggle)

This notebook uses the **Fruit Recognition** dataset by [sshikamaru](https://www.kaggle.com/datasets/sshikamaru/fruit-recognition/data) hosted on Kaggle.

**Manual download (recommended)**
1. Open the dataset page: https://www.kaggle.com/datasets/sshikamaru/fruit-recognition/data
2. Click **Download** (zip, ~600 MB)
3. Extract the archive so the folder structure matches:
   ```
   data/fruit_classification/
   ├── train/   ← 33 class sub-directories
   └── test/
   ```
4. Place the extracted folder at `data/fruit_classification/` relative to the repository root.

**Kaggle CLI (alternative)**  
If you have the [Kaggle API](https://github.com/Kaggle/kaggle-api) set up (`pip install kaggle` + `~/.kaggle/kaggle.json`), run the cell below to download and extract the dataset automatically.


In [14]:
# Download the Fruit Recognition dataset via the Kaggle CLI
# Prerequisites:
#   pip install kaggle
#   Place your kaggle.json API token at ~/.kaggle/kaggle.json
#   (generate it at https://www.kaggle.com/settings → API → Create New Token)

# Uncomment and run to download + extract the dataset:
# import subprocess, sys
# subprocess.run(
#     [
#         sys.executable, "-m", "pip", "install", "--quiet", "kaggle"
#     ],
#     check=True,
# )
# subprocess.run(
#     [
#         "kaggle", "datasets", "download",
#         "-d", "sshikamaru/fruit-recognition",
#         "--unzip",
#         "-p", "../../data/fruit_classification",
#     ],
#     check=True,
# )
# print("Dataset downloaded to data/fruit_classification/")

# Verify the expected layout after download
from pathlib import Path
_root = Path("../../data/fruit_classification")
for _split in ("train", "test"):
    _n = sum(1 for _ in (_root / _split).rglob("*.jpg")) if (_root / _split).exists() else 0
    status = f"{_n} images" if _n else "NOT FOUND"
    print(f"  {_split:6s}: {status}")


  train : 13493 images
  test  : 5641 images


In [15]:
import mata
from pathlib import Path
import numpy as np

print(f"MATA version: {mata.__version__}")

# Root of the fruit classification dataset
DATA_ROOT = Path("../../data/fruit_classification")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"

status = "OK" if TRAIN_DIR.exists() else "NOT FOUND"
print(f"  fruit_classification/train : {status}")
status = "OK" if TEST_DIR.exists() else "NOT FOUND"
print(f"  fruit_classification/test  : {status}")


MATA version: 1.9.5
  fruit_classification/train : OK
  fruit_classification/test  : OK


In [16]:
# Choose 6 visually distinct fruit classes for the gallery
GALLERY_CLASSES = [
    "Apple Braeburn",
    "Banana",
    "Strawberry",
    "Orange",
    "Kiwi",
    "Watermelon",
]

# Verify all selected classes exist in the dataset
for cls in GALLERY_CLASSES:
    d = TRAIN_DIR / cls
    n = len(list(d.glob("*.jpg"))) if d.exists() else 0
    print(f"  {cls:20s}: {n} train images")

# Output directory
runs_dir = Path("../..")/"runs"/"recognition"
runs_dir.mkdir(parents=True, exist_ok=True)
print(f"\nOutput directory: {runs_dir.resolve()}")

  Apple Braeburn      : 394 train images
  Banana              : 392 train images
  Strawberry          : 394 train images
  Orange              : 384 train images
  Kiwi                : 373 train images
  Watermelon          : 380 train images

Output directory: D:\Documents\OneDrive\Code\mtp\datamata_io\mata\runs\recognition


## Step 1 — Build the Gallery

For each class we embed 3 training images with CLIP and average the vectors to form a **class centroid**. This single representative vector is then enrolled in the `Gallery`.

The `Gallery` L2-normalises all vectors on insertion, so cosine similarity equals the dot product during search.

In [17]:
# Load the CLIP embedding model once
encoder = mata.load("embed", "openai/clip-vit-base-patch32")
print(f"Encoder loaded: {encoder}")

[INFO] Loading embed model from huggingface: openai/clip-vit-base-patch32
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceReIDAdapter with device=cuda, threshold=0.3
[INFO] Loading ReID encoder: openai/clip-vit-base-patch32 (arch=clip)


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 23106.55it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder loaded: <mata.adapters.embed_adapter.EmbedAdapter object at 0x000002041F5DA050>


In [18]:
# Build the gallery — embed 3 training images per class, take the centroid
from mata.core.artifacts.image import Image as MataImage

IMAGES_PER_CLASS = 3

gallery = mata.Gallery(similarity_thresh=0.5)

for cls in GALLERY_CLASSES:
    class_dir = TRAIN_DIR / cls
    images = sorted(class_dir.glob("*.jpg"))[:IMAGES_PER_CLASS]
    if not images:
        print(f"  WARNING: no images found for '{cls}'")
        continue

    # Embed each image and collect vectors
    vecs = [encoder.embed(MataImage.from_path(str(img_path))) for img_path in images]
    # vecs is a list of (1, D) arrays — stack and average
    centroid = np.mean(np.vstack(vecs), axis=0).astype(np.float32)

    idx = gallery.add(cls, centroid)
    print(f"  Enrolled '{cls}' at index {idx} (from {len(images)} images)")

print(f"\nGallery: {gallery}")


  Enrolled 'Apple Braeburn' at index 0 (from 3 images)
  Enrolled 'Banana' at index 1 (from 3 images)
  Enrolled 'Strawberry' at index 2 (from 3 images)
  Enrolled 'Orange' at index 3 (from 3 images)
  Enrolled 'Kiwi' at index 4 (from 3 images)
  Enrolled 'Watermelon' at index 5 (from 3 images)

Gallery: Gallery(size=6, unique_labels=6, thresh=0.5)


## Step 2 — Save & Load the Gallery

`gallery.save()` persists the gallery to a `.npz` file using `allow_pickle=False` (secure by default).  
`Gallery.load()` restores the full gallery including the similarity threshold.

In [19]:
# Save
gallery_path = str(runs_dir / "fruit_gallery.npz")
gallery.save(gallery_path)
print(f"Saved gallery to: {gallery_path}")

# Reload from disk
loaded_gallery = mata.Gallery.load(gallery_path)
print(f"Loaded gallery: {loaded_gallery}")
assert loaded_gallery.size == gallery.size, "Gallery size mismatch after reload!"
print("Roundtrip check passed.")

Saved gallery to: ..\..\runs\recognition\fruit_gallery.npz
Loaded gallery: Gallery(size=6, unique_labels=6, thresh=0.5)
Roundtrip check passed.


## Step 3 — Query a Single Image

`mata.run("recognize", image, gallery=gallery, top_k=3)` is the one-liner API:

1. Embeds the query image with the default embed model
2. Runs cosine similarity search against the gallery
3. Returns a `Matches` artifact

Pass `model=` to override the embedding model.

In [33]:
# Pick a held-out training image from the "Strawberry" class
# (skip the first IMAGES_PER_CLASS images already used for gallery building)

strawberry_images = sorted((TRAIN_DIR / "Lemon").glob("*.jpg"))
query_path = strawberry_images[IMAGES_PER_CLASS]   # first image not in the gallery
query_path

WindowsPath('../../data/fruit_classification/train/Lemon/Lemon_100.jpg')

In [34]:

print(f"Query image: {query_path.name}")
result = mata.run(
    "recognize",
    str(query_path),
    model="openai/clip-vit-base-patch32",
    gallery=loaded_gallery,
    top_k=3,
)

# Auto-display the Matches artifact as a rich HTML table
result


Query image: Lemon_100.jpg
[INFO] Loading embed model from huggingface: openai/clip-vit-base-patch32
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceReIDAdapter with device=cuda, threshold=0.3
[INFO] Loading ReID encoder: openai/clip-vit-base-patch32 (arch=clip)


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 27114.09it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Instance ID,Label,Similarity,Top-K
query,Orange,54.4%,2


In [36]:
# Inspect the result programmatically
entry = result.entries[0]   # single entry for a single-image query

print(f"Query image: {query_path.name}")
print(f"Predicted label : {entry.label}")
print(f"Similarity      : {entry.similarity:.1%}")
print()
print("Top-3 matches:")
for m in entry.all_matches:
    print(f"  {m['label']:20s}  {m['similarity']:.3f}")

Query image: Lemon_100.jpg
Predicted label : Orange
Similarity      : 54.4%

Top-3 matches:
  Orange                0.544
  Kiwi                  0.511


## Step 4 — Batch Evaluation

Query one held-out training image per enrolled class (skipping the images used to build the gallery) and measure recognition accuracy.

> **Note:** The Kaggle dataset's `test/` folder contains flat, unlabelled images. We use training images beyond the gallery set as our held-out queries instead.


In [37]:
correct = 0
total   = 0

print(f"{'Class':20s}  {'Predicted':20s}  {'Similarity':>10s}  Match")
print("-" * 70)

for true_label in GALLERY_CLASSES:
    train_images = sorted((TRAIN_DIR / true_label).glob("*.jpg"))
    # Skip the first IMAGES_PER_CLASS images (already used for gallery)
    query_images = train_images[IMAGES_PER_CLASS:]
    if not query_images:
        print(f"  WARNING: no held-out images for '{true_label}'")
        continue

    query = str(query_images[0])
    res = mata.run(
        "recognize",
        query,
        model="openai/clip-vit-base-patch32",
        gallery=loaded_gallery,
        top_k=1,
        threshold=0.0,          # return best match regardless of threshold
    )

    predicted = res.entries[0].label
    similarity = res.entries[0].similarity
    is_correct = predicted == true_label
    correct += int(is_correct)
    total   += 1

    tick = "OK" if is_correct else "WRONG"
    print(f"{true_label:20s}  {predicted:20s}  {similarity:>10.1%}  {tick}")

print("-" * 70)
print(f"Accuracy: {correct}/{total} = {correct/total:.1%}")


Class                 Predicted             Similarity  Match
----------------------------------------------------------------------
[INFO] Loading embed model from huggingface: openai/clip-vit-base-patch32
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceReIDAdapter with device=cuda, threshold=0.3
[INFO] Loading ReID encoder: openai/clip-vit-base-patch32 (arch=clip)


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 28940.78it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Apple Braeburn        Apple Braeburn             56.4%  OK
[INFO] Loading embed model from huggingface: openai/clip-vit-base-patch32
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceReIDAdapter with device=cuda, threshold=0.3
[INFO] Loading ReID encoder: openai/clip-vit-base-patch32 (arch=clip)


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 30787.01it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Banana                Banana                     70.7%  OK
[INFO] Loading embed model from huggingface: openai/clip-vit-base-patch32
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceReIDAdapter with device=cuda, threshold=0.3
[INFO] Loading ReID encoder: openai/clip-vit-base-patch32 (arch=clip)


...

Orange                Orange                     69.7%  OK
[INFO] Loading embed model from huggingface: openai/clip-vit-base-patch32
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceReIDAdapter with device=cuda, threshold=0.3
[INFO] Loading ReID encoder: openai/clip-vit-base-patch32 (arch=clip)


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 24538.19it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Kiwi                  Kiwi                       53.2%  OK
[INFO] Loading embed model from huggingface: openai/clip-vit-base-patch32
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceReIDAdapter with device=cuda, threshold=0.3
[INFO] Loading ReID encoder: openai/clip-vit-base-patch32 (arch=clip)


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 23872.17it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Watermelon            Watermelon                 55.2%  OK
----------------------------------------------------------------------
Accuracy: 5/6 = 83.3%


## Step 5 — Gallery Utilities

The `Gallery` class provides several convenience methods for managing enrolled identities.

In [38]:
# Inspect gallery contents
print(f"Size           : {gallery.size}")
print(f"Unique labels  : {gallery.unique_labels}")
print()

# Directly searching a vector (without going through mata.run)
# Useful when you already have an embedding
kiwi_images = sorted((TRAIN_DIR / "Kiwi").glob("*.jpg"))
sample_path = kiwi_images[IMAGES_PER_CLASS]                          # held-out image
sample_vec  = encoder.embed(MataImage.from_path(str(sample_path)))  # (1, D) ndarray
query_vec   = sample_vec[0]                                          # (D,) 1-D vector

matches = gallery.search(query_vec, top_k=3, threshold=0.0)
print("Direct gallery.search() for a Kiwi image:")
for m in matches:
    print(f"  {m.label:20s}  {m.similarity:.3f}")

# Serialise the gallery to JSON (handy for quick inspection)
import json
gallery_dict = gallery.to_dict()
print(f"\nto_dict() keys: {list(gallery_dict.keys())}")


Size           : 6
Unique labels  : ['Apple Braeburn', 'Banana', 'Strawberry', 'Orange', 'Kiwi', 'Watermelon']

Direct gallery.search() for a Kiwi image:
  Kiwi                  0.532
  Watermelon            0.457
  Apple Braeburn        0.449

to_dict() keys: ['embeddings', 'labels', 'similarity_thresh', 'size']


## Step 6 — Graph Pipeline Pattern

For per-ROI recognition (e.g. recognising detected objects), compose
`Detect >> ExtractROIs >> Embed >> GalleryMatchNode`:

```python
from mata.core.graph import Graph
from mata.nodes import Detect, ExtractROIs, Embed, GalleryMatchNode

graph = (
    Graph("fruit_recognition")
    .then(Detect(using="detector"))
    .then(ExtractROIs(src_dets="dets"))
    .then(Embed(using="encoder"))
    .then(GalleryMatchNode(gallery=gallery, top_k=1, threshold=0.5))
)

result = mata.infer(
    graph,
    image="image.jpg",
    providers={"detector": detector, "encoder": encoder},
)
matches = result["matches"]  # Matches artifact with one entry per ROI
```

The code below builds and describes the graph (no model download required):

In [39]:
from mata.nodes import GalleryMatchNode

# Build the GalleryMatchNode — inspect its interface
match_node = GalleryMatchNode(gallery=loaded_gallery, top_k=1, threshold=0.5)
print(f"Node  : {match_node}")
print(f"Inputs: {list(match_node.inputs.keys())}")
print(f"Outputs: {list(match_node.outputs.keys())}")

Node  : GalleryMatchNode(gallery_size=6, top_k=1, threshold=0.5)
Inputs: ['embeddings']
Outputs: ['matches']


## Notes

**Result inspection**
```python
# Single-image convenience API
result = mata.run("recognize", image, gallery=gallery, top_k=3)
entry  = result.entries[0]      # one entry for single-image queries
entry.label                      # best-matching gallery label
entry.similarity                 # cosine similarity in [0, 1]
entry.all_matches                # list of top-k GalleryMatch dicts
result.to_json()                 # full JSON export
result.to_dict()                 # dict export
```

**Gallery management**
```python
gallery.add("alice", embedding)          # enrol one identity
gallery.add_many(labels, matrix)         # bulk enrol
gallery.remove("alice")                  # remove by label
gallery.search(vec, top_k=5)            # direct similarity search
gallery.search_batch(vecs, top_k=5)     # batch search
gallery.save("gallery.npz")             # persist
mata.Gallery.load("gallery.npz")        # restore
```

**CLI usage**
```bash
# Save the gallery first
# Then recognise from the command line:
mata recognize image.jpg --gallery runs/recognition/fruit_gallery.npz --model openai/clip-vit-base-patch32 --top-k 3
```

**Threshold tuning**  
Set `similarity_thresh` when constructing the `Gallery` (default 0.5).  
Pass `threshold=` at query time to override for that call.